In [ ]:
# PREDICTIVE MODELLING OF CRIME ARREST OUTCOMES USING BIG DATA ANALYTICS AND
# MACHINE LEARNING TECHNIQUES
#===============================================================================

In [ ]:
# "SECTIONS" REPRESENT THE 4 EXPECTED NOTEBOOKS

In [ ]:
# ==============================================================================
# SECTION 0: SETUPS
# STEP 0.1: SETTING UP GOOGLE COLAB
# ==============================================================================
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.3.0/spark-3.3.0-bin-hadoop3.tgz
!tar xf spark-3.3.0-bin-hadoop3.tgz

In [ ]:
# STEP 0.2: INSTALLING JAVA
#===============================================================================
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.3.0-bin-hadoop3"

In [ ]:
# STEP 0.3: INSTALLING PYSPARK
#===============================================================================
!pip install pyspark

In [ ]:
# STEP 0.4: IMPORTATIONS OF LIBRARIES, MODELS AND PACKAGES
#===============================================================================
import os, csv
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder,
    VectorAssembler, Imputer, StandardScaler
)
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.stat import Correlation

In [ ]:
# STEP 0.5: VERIFYING SPARK SESSION; aimed at getting a Spark UI screenshots.
# unfortunately, all the links were invalid.
#===============================================================================
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("CrimeData") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

In [ ]:
#===============================================================================
# SECTION 1: DATA INGESTION AND CLEANING
#===============================================================================

In [ ]:
# STEP 1: MOUNTING GOOGLE DRIVE
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# STEP 2: UPLOADING CSV FROM MY GOOGLE DRIVE
# ==============================================================================
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CrimeData").getOrCreate()

df_spark = spark.read.csv('/content/drive/MyDrive/datasets/Crimes_-_2001_to_Present.csv',
                          header=True, inferSchema=True)

df_spark.show(5)

+--------+-----------+--------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------------------+
|      ID|Case Number|                Date|               Block|IUCR|        Primary Type|         Description|Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|          Updated On|    Latitude|    Longitude|            Location|
+--------+-----------+--------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------------------+
|13311263|   JG503434|07/29/2022 03:39:...|     023XX S TROY ST|1582|OFFENSE INVOLVING...|   CHILD PORNOGRAPHY|           RE

In [ ]:
# STEP 3: GETTING A SPARK UI SCREENSHOT
#===============================================================================
from google.colab.output import eval_js
eval_js("google.colab.kernel.proxyPort(4050)")

'https://4050-m-s-2mf3j5zvmimdm-d.us-east1-0.prod.colab.dev'

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("CrimeData") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

In [ ]:
# STEP 4: VERIFYING AND ANALYSING MY DATA STRUCTURE AND PATHWAY
# ==============================================================================
df = spark.read.csv(
    '/content/drive/MyDrive/datasets/Crimes_-_2001_to_Present.csv',
    header=True,
    inferSchema=True
)

In [ ]:
df.show(5)

+--------+-----------+--------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------------------+
|      ID|Case Number|                Date|               Block|IUCR|        Primary Type|         Description|Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|          Updated On|    Latitude|    Longitude|            Location|
+--------+-----------+--------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------------------+
|13311263|   JG503434|07/29/2022 03:39:...|     023XX S TROY ST|1582|OFFENSE INVOLVING...|   CHILD PORNOGRAPHY|           RE

In [ ]:
# SECTION 1.2: DATA CLEANING
# ==============================================================================
# STEP 1: STANDADIZING COLUMN NAMES
#===============================================================================
for c in df.columns:
    df = df.withColumnRenamed(
        c, c.strip().lower().replace(" ", "_")
    )

df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- case_number: string (nullable = true)
 |-- date: string (nullable = true)
 |-- block: string (nullable = true)
 |-- iucr: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: boolean (nullable = true)
 |-- domestic: boolean (nullable = true)
 |-- beat: integer (nullable = true)
 |-- district: integer (nullable = true)
 |-- ward: integer (nullable = true)
 |-- community_area: integer (nullable = true)
 |-- fbi_code: string (nullable = true)
 |-- x_coordinate: integer (nullable = true)
 |-- y_coordinate: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- updated_on: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location: string (nullable = true)



In [ ]:
# STEP 2: STABILIZING DATE AND TIME FEATURES
#===============================================================================
from pyspark.sql.functions import col, to_timestamp, coalesce

df = df.withColumn(
    "Date",
    coalesce(
        to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a"),
        to_timestamp(col("Date"), "MM/dd/yyyy HH:mm:ss"),
        to_timestamp(col("Date"), "yyyy-MM-dd HH:mm:ss")
    )
)
df = df.dropna(subset=["date"])

from pyspark.sql.functions import hour, dayofweek, month, when
df = (
    df
    .withColumn("hour", hour("date"))
    .withColumn("day_of_week", dayofweek("date"))
    .withColumn("month", month("date"))
    .withColumn(
        "is_weekend",
        when(dayofweek("date").isin([1, 7]), 1).otherwise(0)
    )
)

In [ ]:
# STEP 3: CASTING TARGET VARIABLE (ARREST)
#===============================================================================
df = df.withColumn("arrest", col("arrest").cast("int"))
df.groupBy("arrest").count().show()

+------+-------+
|arrest|  count|
+------+-------+
|     1|2137134|
|     0|6361180|
+------+-------+



In [ ]:
# STEP 4: CHECKING DATA (To be sure that all features have been transformed and saved safely)
#===============================================================================
df.show(5, truncate=False)
df.printSchema()
print("Total records:", df.count())

+--------+-----------+-------------------+-----------------------+----+--------------------------+------------------------------+--------------------------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+----+-----------+-----+----------+
|id      |case_number|Date               |block                  |iucr|primary_type              |description                   |location_description                  |arrest|domestic|beat|district|ward|community_area|fbi_code|x_coordinate|y_coordinate|year|updated_on            |latitude    |longitude    |location                     |hour|day_of_week|month|is_weekend|
+--------+-----------+-------------------+-----------------------+----+--------------------------+------------------------------+--------------------------------------+------+--------+----+--------+----+--------------+--------+------------+------------+-

In [ ]:
# STEP 5: WRITING PARQUET
#===============================================================================
df.write.mode("overwrite").parquet(PARQUET_PATH)

print("Parquet written to:", PARQUET_PATH)

Parquet written to: /content/crime_project/processed_parquet


In [ ]:
#===============================================================================
# SECTION 2: FEATURE ENGINEERING AND EXPLORATORY DATA ANALYSIS
#===============================================================================

# STEP 1: VERIFYING PARQUET PATHWAY/ READING
#===============================================================================
df_parquet = spark.read.parquet(PARQUET_PATH)
df_parquet.printSchema()
print("Rows in Parquet:", df_parquet.count())

root
 |-- id: integer (nullable = true)
 |-- case_number: string (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- block: string (nullable = true)
 |-- iucr: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: integer (nullable = true)
 |-- domestic: boolean (nullable = true)
 |-- beat: integer (nullable = true)
 |-- district: integer (nullable = true)
 |-- ward: integer (nullable = true)
 |-- community_area: integer (nullable = true)
 |-- fbi_code: string (nullable = true)
 |-- x_coordinate: integer (nullable = true)
 |-- y_coordinate: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- updated_on: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: i

In [ ]:
# STEP 2: DROPPING NON-PREDICTIVE COLUMNS
#===============================================================================
cols_to_drop = [
    "id",
    "case_number",
    "block",
    "iucr",
    "description",
    "beat",
    "x_coordinate",
    "y_coordinate",
    "latitude",
    "longitude",
    "location",
    "fbi_code",
    "updated_on"
]

df = df.drop(*cols_to_drop)

In [ ]:
# STEP 3: REMOVING ROWS WITH MISSING CRITICAL VALUES
#===============================================================================
df = df.dropna(subset=[
    "primary_type",
    "location_description",
    "district",
    "ward",
    "community_area",
    "arrest"
])

In [ ]:
# STEP 4: DEFINING LABEL AND FEATURE COLUMNS
#===============================================================================
label_col = "arrest"

categorical_cols = [
    "primary_type",
    "location_description",
    "district",
    "ward",
    "community_area"
]

numeric_cols = [
    "year",
    "hour",
    "day_of_week",
    "month",
    "is_weekend"
]
binary_cols = ["domestic"]

In [ ]:
#===============================================================================
# SECTION 2.2: EXPLORATORY DATA ANALYSIS (EDA)
#===============================================================================

# EDA_1: DATASET AND TARGET DISTRIBUTION
#===============================================================================
print("Records after cleaning:", df.count())
df.groupBy("arrest").count().show()
from pyspark.sql.functions import avg, col, count

Records after cleaning: 114088
+------+-----+
|arrest|count|
+------+-----+
|     1|19740|
|     0|94348|
+------+-----+



In [ ]:
# EDA_2: ARREST RATE BY HOUR (FIXED WORKING)
#===============================================================================
df.groupBy("hour") \
  .agg(
      avg(col("arrest").cast("int")).alias("arrest_rate"),
      count("*").alias("total_cases")
  ) \
  .orderBy("hour") \
  .show(24)

+----+-------------------+-----------+
|hour|        arrest_rate|total_cases|
+----+-------------------+-----------+
|   0| 0.1267212325469427|      10385|
|   1|0.20048899755501223|       4090|
|   2| 0.1812306827760607|       3559|
|   3|0.18054630246502332|       3002|
|   4|0.16982758620689656|       2320|
|   5|0.16466789667896678|       2168|
|   6|  0.164257555847569|       2283|
|   7|0.14857554994590697|       2773|
|   8|0.13754940711462452|       3795|
|   9|0.13755240566979438|       5009|
|  10|0.18845573982339006|       4643|
|  11|0.21138388007820985|       4603|
|  12|0.14696876913655849|       6532|
|  13|0.18194668820678514|       4952|
|  14| 0.1802875520242149|       5286|
|  15| 0.1654443117857752|       5863|
|  16| 0.1958513396715644|       5785|
|  17|0.17667658903869726|       5711|
|  18|0.17034102652428523|       5806|
|  19|0.18207782934666192|       5602|
|  20|0.19458762886597938|       5432|
|  21|0.19840796019900497|       5025|
|  22|0.19344194679315324

In [ ]:
# EDA_3: ARREST RATE BY DISTRICT
#===============================================================================
df.groupBy("district") \
  .agg(
      avg(col("arrest").cast("int")).alias("arrest_rate"),
      count("*").alias("total_cases")
  ) \
  .orderBy(col("arrest_rate").desc()) \
  .show(10)

+--------+-------------------+-----------+
|district|        arrest_rate|total_cases|
+--------+-------------------+-----------+
|      11| 0.3218047253060063|       7026|
|      31|               0.25|          4|
|      10|0.23291606813051507|       4873|
|      15|0.23092219736515038|       4023|
|       7|0.21790930053804766|       5204|
|       5|0.18707412760685527|       4843|
|       9| 0.1847063169560399|       5414|
|       1|0.18449053201082055|       5545|
|      16| 0.1743428984808295|       4147|
|       3| 0.1734475374732334|       5604|
+--------+-------------------+-----------+
only showing top 10 rows


In [ ]:
# EDA_4: DOMESTIC VS NON-DOMESTIC CRIMES
#===============================================================================
df.groupBy("domestic") \
  .agg(
      avg(col("arrest").cast("int")).alias("arrest_rate"),
      count("*").alias("total_cases")
  ) \
  .show()

+--------+-------------------+-----------+
|domestic|        arrest_rate|total_cases|
+--------+-------------------+-----------+
|    true|0.18763431972226816|      18147|
|   false|0.17026088950500828|      95941|
+--------+-------------------+-----------+



In [ ]:
# EDA_5: ARREST RATE BY CRIME TYPE
#===============================================================================
df.groupBy("primary_type") \
  .agg(
      avg(col("arrest").cast("int")).alias("arrest_rate"),
      count("*").alias("total_cases")
  ) \
  .orderBy(col("total_cases").desc()) \
  .show(10)

+-------------------+-------------------+-----------+
|       primary_type|        arrest_rate|total_cases|
+-------------------+-------------------+-----------+
|              THEFT|0.05450833912439194|      23024|
|            BATTERY|0.17616419668878278|      16127|
|           HOMICIDE|0.47687974336906347|      12781|
|    CRIMINAL DAMAGE| 0.0326394502829426|       9896|
|MOTOR VEHICLE THEFT|0.03830624288228595|       9659|
| DECEPTIVE PRACTICE|0.03944678292242935|       8315|
|            ASSAULT|0.11049451227031694|       8109|
|      OTHER OFFENSE|0.15813953488372093|       5805|
|            ROBBERY|0.08825910931174089|       4940|
|           BURGLARY|0.08687381103360811|       3154|
+-------------------+-------------------+-----------+
only showing top 10 rows


In [ ]:
# EDA_6: ARREST RATE BY CRIME TYPE
#===============================================================================
df.groupBy("primary_type") \
  .agg(
      avg(col("arrest").cast("int")).alias("arrest_rate"),
      count("*").alias("total_cases")
  ) \
  .orderBy(col("total_cases").desc()) \
  .show(10)

+-----------+-------------------+-----------+
|day_of_week|        arrest_rate|total_cases|
+-----------+-------------------+-----------+
|          1|0.25106784623014283|      62509|
|          2|0.26639876352395675|      64700|
|          3| 0.2764058315977388|      63859|
|          4| 0.2722079450103725|      64594|
|          5| 0.2687726807481158|      64482|
|          6|  0.259580869539314|      67139|
|          7| 0.2558798044191372|      64628|
+-----------+-------------------+-----------+



In [ ]:
# EDA_7: CORRELATION ANALYSIS WITH NUMERIC TIME FEATURES
#===============================================================================
numeric_eda_cols = ["year", "hour", "day_of_week", "month", "is_weekend"]

# Creating a numeric version of the target variable
df_corr = df.withColumn("arrest_int", col("arrest").cast("int"))

for col_name in numeric_eda_cols:
    corr_value = df_corr.stat.corr(col_name, "arrest_int")
    print(f"Correlation between {col_name} and arrest:", corr_value)


Correlation between year and arrest: -0.21032288334048985
Correlation between hour and arrest: 0.03414175561747853
Correlation between day_of_week and arrest: -0.008106109663124025
Correlation between month and arrest: -0.16031120564483167
Correlation between is_weekend and arrest: -0.001111427550948037


In [ ]:
#===============================================================================
# SECTION 3: MODEL TRAINING AND TESTING (4 MODELS)
#===============================================================================
# STEP 1: Train-test split

train_df, test_df = df_parquet.randomSplit([0.8, 0.2], seed=42)

print("Training records:", train_df.count())
print("Testing records:", test_df.count())

Training records: 361253
Testing records: 90658


In [ ]:
# STEP 2: DEFINING TARGET FEATURES
#===============================================================================
label_col = "arrest"

categorical_cols = [
    "primary_type",
    "location_description",
    "district",
    "ward",
    "community_area"
]

numeric_cols = [
    "year",
    "hour",
    "day_of_week",
    "month",
    "is_weekend"
]
binary_cols = ["domestic"]

In [ ]:
# STEP 3: FEATURE PREPROCESSING PIPELINE
#===============================================================================

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)

# Index categorical variables
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

# Hot-end encoding
encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_ohe"
    )
    for c in categorical_cols
]

# Features Assembling
assembler = VectorAssembler(
    inputCols=[f"{c}_ohe" for c in categorical_cols]
              + numeric_cols
              + binary_cols,
    outputCol="features_raw"
)

# Scale features
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features"
)

preprocess_stages = indexers + encoders + [assembler, scaler]

In [ ]:
# STEP 4: EVALUATION FUNCTION
#===============================================================================

from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

def evaluate_model(predictions, model_name):
    return {
        "model": model_name,
        "accuracy": MulticlassClassificationEvaluator(
            labelCol=label_col, metricName="accuracy"
        ).evaluate(predictions),

        "precision": MulticlassClassificationEvaluator(
            labelCol=label_col, metricName="weightedPrecision"
        ).evaluate(predictions),

        "recall": MulticlassClassificationEvaluator(
            labelCol=label_col, metricName="weightedRecall"
        ).evaluate(predictions),

        "f1": MulticlassClassificationEvaluator(
            labelCol=label_col, metricName="f1"
        ).evaluate(predictions),

        "auc": BinaryClassificationEvaluator(
            labelCol=label_col, metricName="areaUnderROC"
        ).evaluate(predictions)
    }

In [ ]:
# STEP 5: SEQUENTIAL TRAINING OF 4 SUPERVISED MODELS
#===============================================================================

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier
)

In [ ]:
# MODEL 1: Logistic Regression
# ==============================================================================

print("Training Logistic Regression...")

lr_pipeline = Pipeline(
    stages=preprocess_stages + [
        LogisticRegression(labelCol=label_col)
    ]
)

lr_model = lr_pipeline.fit(train_df)
lr_preds = lr_model.transform(test_df)

lr_results = evaluate_model(lr_preds, "Logistic Regression")
lr_results

Training Logistic Regression...


{'model': 'Logistic Regression',
 'accuracy': 0.8571995852544728,
 'precision': 0.8573371082599188,
 'recall': 0.8571995852544728,
 'f1': 0.8460437564656393,
 'auc': 0.8640372375185623}

In [ ]:
# Delete (For memory optimization, as i will no longer be using it)
del lr_model

In [ ]:
# MODEL 2: Decision Tree
# ==============================================================================

print("Training Decision Tree...")

dt_pipeline = Pipeline(
    stages=preprocess_stages + [
        DecisionTreeClassifier(labelCol=label_col)
    ]
)

dt_model = dt_pipeline.fit(train_df)
dt_preds = dt_model.transform(test_df)

dt_results = evaluate_model(dt_preds, "Decision Tree")
dt_results

Training Decision Tree...


{'model': 'Decision Tree',
 'accuracy': 0.8430805885856736,
 'precision': 0.8585158574092714,
 'recall': 0.8430805885856736,
 'f1': 0.8213693484606172,
 'auc': 0.2881523533439826}

In [ ]:
# Safe to delete (no further use)
del dt_model

In [ ]:
# MODEL 3: Random Forest
# ==============================================================================

print("Training Random Forest...")

rf_pipeline = Pipeline(
    stages=preprocess_stages + [
        RandomForestClassifier(
            labelCol=label_col,
            numTrees=50,
            seed=42
        )
    ]
)

rf_model = rf_pipeline.fit(train_df)
rf_preds = rf_model.transform(test_df)

rf_results = evaluate_model(rf_preds, "Random Forest")
rf_results

Training Random Forest...


{'model': 'Random Forest',
 'accuracy': 0.7347945024156721,
 'precision': 0.5399229607802951,
 'recall': 0.7347945024156721,
 'f1': 0.6224633062053881,
 'auc': 0.8291364768099554}

In [ ]:
del rf_model

In [ ]:
# MODEL 4: GRADIENT BOOSTED TREES
# ==============================================================================
from pyspark.ml.classification import GBTClassifier
from pyspark.ml import Pipeline

In [ ]:
print("Training Gradient-Boosted Trees (GBT)...")

gbt_pipeline = Pipeline(
    stages=preprocess_stages + [
        GBTClassifier(
            labelCol=label_col,
            maxIter=20,        # I kept it low as to enhance procesing
            maxDepth=5,        # I needed to avoid deep trees
            stepSize=0.1,
            seed=42
        )
    ]
)

gbt_model = gbt_pipeline.fit(train_df)
gbt_preds = gbt_model.transform(test_df)

gbt_results = evaluate_model(gbt_preds, "Gradient Boosted Trees")
gbt_results

Training Gradient-Boosted Trees (GBT)...


{'model': 'Gradient Boosted Trees',
 'accuracy': 0.8596924706038077,
 'precision': 0.8674437009373568,
 'recall': 0.8596924706038077,
 'f1': 0.8452966617324812,
 'auc': 0.8584232154373748}

In [ ]:
#===============================================================================
# SECTION 4: MODEL EVALUATION AND KMEANS CLUSTERING
#===============================================================================

In [ ]:
# SECTION 4.1: MODEL EVALUATION

# STEP 1: MODEL EVALUATION METRICS
#===============================================================================
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

def evaluate_model(preds, model_name):
    return {
        "model": model_name,
        "accuracy": MulticlassClassificationEvaluator(
            labelCol=label_col, metricName="accuracy"
        ).evaluate(preds),
        "precision": MulticlassClassificationEvaluator(
            labelCol=label_col, metricName="weightedPrecision"
        ).evaluate(preds),
        "recall": MulticlassClassificationEvaluator(
            labelCol=label_col, metricName="weightedRecall"
        ).evaluate(preds),
        "f1": MulticlassClassificationEvaluator(
            labelCol=label_col, metricName="f1"
        ).evaluate(preds),
        "auc": BinaryClassificationEvaluator(
            labelCol=label_col, metricName="areaUnderROC"
        ).evaluate(preds)
    }

In [ ]:
# STEP 2: BUILDING THE FINAL COMPARISON TABLE
#===============================================================================
results_df = spark.createDataFrame([
    lr_results,
    dt_results,
    rf_results,
    gbt_results
])

results_df.show(truncate=False)


+------------------+------------------+------------------+----------------------+------------------+------------------+
|accuracy          |auc               |f1                |model                 |precision         |recall            |
+------------------+------------------+------------------+----------------------+------------------+------------------+
|0.8571995852544728|0.8640372375185623|0.8460437564656393|Logistic Regression   |0.8573371082599188|0.8571995852544728|
|0.8430805885856736|0.2881523533439826|0.8213693484606172|Decision Tree         |0.8585158574092714|0.8430805885856736|
|0.7347945024156721|0.8291364768099554|0.6224633062053881|Random Forest         |0.5399229607802951|0.7347945024156721|
|0.8596924706038077|0.8584232154373748|0.8452966617324812|Gradient Boosted Trees|0.8674437009373568|0.8596924706038077|
+------------------+------------------+------------------+----------------------+------------------+------------------+



In [ ]:
# STEP 3: IDENTIFYING THE BEST MODEL BY AUC
#====================================================================

results_df.orderBy("auc", ascending=False).show(truncate=False)

+------------------+------------------+------------------+----------------------+------------------+------------------+
|accuracy          |auc               |f1                |model                 |precision         |recall            |
+------------------+------------------+------------------+----------------------+------------------+------------------+
|0.8571995852544728|0.8640372375185623|0.8460437564656393|Logistic Regression   |0.8573371082599188|0.8571995852544728|
|0.8596924706038077|0.8584232154373748|0.8452966617324812|Gradient Boosted Trees|0.8674437009373568|0.8596924706038077|
|0.7347945024156721|0.8291364768099554|0.6224633062053881|Random Forest         |0.5399229607802951|0.7347945024156721|
|0.8430805885856736|0.2881523533439826|0.8213693484606172|Decision Tree         |0.8585158574092714|0.8430805885856736|
+------------------+------------------+------------------+----------------------+------------------+------------------+



In [ ]:
# STEP 4: CONFUSION MATRIX (DONE USING THE BEST MODEL)
#===============================================================================
if gbt_model is not None:
    print("Computing confusion matrix for Gradient Boosted Trees...")

    gbt_predictions = gbt_model.transform(test_df)

    confusion_matrix = (
        gbt_predictions
        .groupBy("arrest", "prediction")
        .count()
        .orderBy("arrest", "prediction")
    )

    confusion_matrix.show()

else:
    print("Gradient Boosted Trees model not available due to runtime constraints.")
    print("Confusion matrix omitted — evaluation based on metrics table.")

Computing confusion matrix for Gradient Boosted Trees...
+------+----------+-----+
|arrest|prediction|count|
+------+----------+-----+
|     0|       0.0|65485|
|     0|       1.0| 1130|
|     1|       0.0|11590|
|     1|       1.0|12453|
+------+----------+-----+



In [ ]:
# SECTION 4.2: KMEANS CLUSTERING (UNSUPERVISED MODEL)
#===============================================================================
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
from pyspark.sql.functions import avg, count

print("Running K-Means clustering")

# Sample data representing the scalable data
kmeans_df = df_parquet.sample(
    fraction=0.05,   # 5% sample (defensible and sufficient)
    seed=42
)
print("K-Means sample size:", kmeans_df.count())

kmeans_pipeline = Pipeline(
    stages=preprocess_stages + [
        KMeans(
            featuresCol="features",
            k=5,          # sensible, explainable choice
            seed=42
        )
    ]
)

Running K-Means clustering
K-Means sample size: 22645


In [ ]:
# STEP 2: POST CLUSTER SUMMARY
#===============================================================================
clustered_df = kmeans_pipeline.fit(df).transform(df)
clustered_df.printSchema()
clustered_df.show(5)

root
 |-- Date: timestamp (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: integer (nullable = true)
 |-- domestic: boolean (nullable = true)
 |-- district: integer (nullable = true)
 |-- ward: integer (nullable = true)
 |-- community_area: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- is_weekend: integer (nullable = false)
 |-- primary_type_idx: double (nullable = false)
 |-- location_description_idx: double (nullable = false)
 |-- district_idx: double (nullable = false)
 |-- ward_idx: double (nullable = false)
 |-- community_area_idx: double (nullable = false)
 |-- primary_type_ohe: vector (nullable = true)
 |-- location_description_ohe: vector (nullable = true)
 |-- district_ohe: vector (nullable = true)
 |-- ward_ohe: vector (nullable = true)
 |-- community_area_ohe:

In [ ]:
from pyspark.sql.functions import avg

cluster_arrest_rate = (
    clustered_df
    .groupBy("prediction")
    .agg(avg("arrest").alias("arrest_rate"))
)

cluster_arrest_rate.show()

+----------+-------------------+
|prediction|        arrest_rate|
+----------+-------------------+
|         1| 0.1399662731871838|
|         3|0.15236231478892928|
|         4|0.10173548773189707|
|         2|0.18973372325688778|
|         0|0.16110699283419816|
+----------+-------------------+



In [ ]:
#===============================================================================
# FINAL RESULTS AND TABLEAU EXPORT DATA SHEETS
#===============================================================================

# 1. MODEL COMPASISON TABLE
#===============================================================================

import os

tableau_path = "/content/tableau_outputs"
os.makedirs(tableau_path, exist_ok=True)

model_comparison_pd = results_df.toPandas()

model_comparison_pd.to_csv(
    f"{tableau_path}/model_comparison.csv",
    index=False
)
print("Model comparison exported.")

Model comparison exported.


In [ ]:
# 2. PREDICTION-LEVEL DATASET FOR TABLEAU (FROM GBT)
#===============================================================================

gbt_predictions = gbt_model.transform(test_df)

gbt_predictions.select(
    "arrest",
    "prediction",
    "probability",
    "primary_type",
    "district",
    "hour",
    "day_of_week",
    "month"
).sample(
    fraction=0.1,      # I reduced the size for Tableau
    seed=42
).toPandas().to_csv(
    "tableau_gbt_predictions.csv",
    index=False
)
print("Prediction-level dataset exported.")

Prediction-level dataset exported.


In [ ]:
from pyspark.sql.functions import avg

gbt_arrest_rate = (
    gbt_predictions
    .groupBy("prediction")
    .agg(avg("arrest").alias("arrest_rate"))
)

gbt_arrest_rate.show()

+----------+------------------+
|prediction|       arrest_rate|
+----------+------------------+
|       0.0| 0.150373013298735|
|       1.0|0.9168077744239123|
+----------+------------------+



In [ ]:
# 3. CLUSTERING OUTPUT
#===============================================================================

# Fitting KMeans and creating clustered_df
kmeans_model = kmeans_pipeline.fit(kmeans_df)
clustered_df = kmeans_model.transform(kmeans_df)

print("clustered_df recreated")

clustered_df recreated


In [ ]:
clustered_tableau = clustered_df.select(
    "prediction",
    "arrest",
    "primary_type",
    "district",
    "hour"
)

clustered_tableau_pd = clustered_tableau.sample(
    fraction=0.1, seed=42
).toPandas()

clustered_tableau_pd.to_csv(
    f"{tableau_path}/crime_clusters_sample.csv",
    index=False
)

print("Clustering data exported.")

Clustering data exported.


In [ ]:
!ls /content

crime_project		      sample_data		   tableau_outputs
Crimes_-_2001_to_Present.csv  tableau_gbt_predictions.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')